# Bagging과 Random Forest

Bagging은 같은 유형의 기본 모델을 여러 개 만들고, 각 모델이 서로 조금씩 다른 데이터를 보게 하여 예측을 결합하는 방식이다.

![https://medium.com/@brijesh_soni/boost-your-machine-learning-models-with-bagging-a-powerful-ensemble-learning-technique-692bfc4d1a51](https://miro.medium.com/v2/resize:fit:1400/format:webp/1*a6hnuJ8WM37mLimHfMORmQ.png)

1. 하나의 모델이 특정 학습 데이터에 과하게 맞춰지는 문제를 줄인다.
2. 여러 모델의 예측을 평균 또는 다수결로 묶어 예측을 더 안정적으로 만든다.
3. 특히 분산이 큰 모델(데이터가 조금만 바뀌어도 모델이 크게 바뀌는 모델-예:결정트리)의 성능을 안정화하는 데 효과적이다.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## 01. RandomForestClassifier

Random Forest는 여러 개의 결정트리를 만든 뒤 그 결과를 결합하는 모델이다.

분류에서는:
- 각 트리가 클래스를 예측한다.
- 최종적으로는 다수결 또는 클래스 확률 평균을 바탕으로 최종 클래스를 정한다.

특징
- 결정트리 1개는 해석은 쉽지만 데이터에 따라 흔들리기 쉽다.
- 트리를 여러 개 만들고 평균적인 판단을 하게 하면 예측이 더 안정적이 된다.
- 즉, Bagging은 성능 향상뿐 아니라 변동성 감소가 핵심이다.

In [2]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

# 데이터 로드
X, y = load_breast_cancer(return_X_y=True)

# 훈련/테스트 분리
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

(455, 30) (455,)
(114, 30) (114,)


In [3]:
from sklearn.ensemble import RandomForestClassifier

# 모델 학습
rf_clf = RandomForestClassifier(
    n_estimators=100,           # 만들 트리 개수
    max_depth=3,                # 각 트리의 최대 깊이 -> 과적합 경향과 일반화 성능을 보기 위해 적당히 제한
    random_state=42
)

rf_clf.fit(X_train, y_train)

print('학습셋 정확도 : ', rf_clf.score(X_train, y_train))
print('테스트셋 정확도 : ', rf_clf.score(X_test, y_test))

학습셋 정확도 :  0.9802197802197802
테스트셋 정확도 :  0.956140350877193


### 부트스트랩 샘플링

부트스트랩은 복원추출이다. 즉, 한 번 뽑은 샘플도 다시 뽑힐 수 있다.

그래서 각 트리는:
- 어떤 샘플은 여러 번 학습하고
- 어떤 샘플은 아예 보지 못한다

이 차이가 트리마다 서로 다른 학습 경험을 만들고, 그 결과 여러 트리의 예측이 완전히 같아지지 않게 된다.

바로 이 차이가 Bagging의 힘이다.

In [6]:
print('트리 개수 : ', len(rf_clf.estimators_))
print('부트스트랩 샘플 정보 개수 : ', len(rf_clf.estimators_samples_))

# 앞에서 3개의 트리만 확인
for tree_idx, sample_index in enumerate(rf_clf.estimators_samples_[:3]):
    unique_count = len(np.unique(sample_index))
    duplicate_count = len(sample_index) - unique_count
    print(f'트리:{tree_idx}, 전체 샘플 수:{len(sample_index)}, 유니크 샘플 수:{unique_count}, 중복 샘플 수:{duplicate_count}')
    print(f'앞부분 인덱스 예시:{sample_index[:20]}')

트리 개수 :  100
부트스트랩 샘플 정보 개수 :  100
트리:0, 전체 샘플 수:455, 유니크 샘플 수:281, 중복 샘플 수:174
앞부분 인덱스 예시:[ 41 378 332 130 393 231  92 199 403 370 120 445  28 233  84   6 397 324
 401 338]
트리:1, 전체 샘플 수:455, 유니크 샘플 수:286, 중복 샘플 수:169
앞부분 인덱스 예시:[174 151 279  31 357 234   3 425 303 202  83 200 163 109 355 104  72 274
  71 249]
트리:2, 전체 샘플 수:455, 유니크 샘플 수:279, 중복 샘플 수:176
앞부분 인덱스 예시:[342 430 288  71 231 250 168 188 373 440  18  53 142  52 387 120 422 126
 379  83]


### 특성 중요도 해석 

특성 중요도는 이 특성이 예측에 얼마나 자주, 크게 기여했는가를 보여주는 지표이다.
하지만 이것을 곧바로 인과관계로 해석하면 안 된다.

즉,
- 중요도가 높다 = 모델이 자주 활용했다
- 중요도가 높다 ≠ 원인이다

In [7]:
# 특성 중요도 확인

cancer = load_breast_cancer()

feature_importance_ser = pd.Series(
    rf_clf.feature_importances_,
    index=cancer.feature_names
).sort_values(ascending=False)

feature_importance_ser

worst area                 0.154355
worst concave points       0.135059
mean concave points        0.103214
worst radius               0.102524
mean perimeter             0.077128
mean radius                0.073345
worst perimeter            0.070119
mean concavity             0.061262
mean area                  0.056625
worst concavity            0.039011
area error                 0.031337
worst compactness          0.016466
radius error               0.013414
mean compactness           0.011368
worst texture              0.007727
perimeter error            0.007655
worst smoothness           0.007325
mean texture               0.005796
compactness error          0.004056
texture error              0.003571
mean smoothness            0.003261
worst symmetry             0.002891
concavity error            0.002663
worst fractal dimension    0.002613
concave points error       0.002088
symmetry error             0.001258
fractal dimension error    0.001191
smoothness error           0

## 02. 일반화 성능 확인

Random Forest도 결국 모델이므로 한 번의 train/test split만 보는 것보다 교차검증을 통해 조금 더 안정적으로 확인하는 것이 좋다.

한 번의 train/test 분할 결과는 우연이 섞일 수 있다.

교차검증은 데이터를 여러 번 나누어 평가하므로, 모델이 특정 분할에만 우연히 잘 맞은 것인지 아닌지를 확인하는 데 도움이 된다.

즉,
- 단일 점수 = 한 번의 결과
- 교차검증 평균 = 조금 더 안정적인 성능 추정

In [8]:
from sklearn.model_selection import cross_validate

rf_clf_cv = RandomForestClassifier(random_state=42, max_depth=10)

results = cross_validate(
    rf_clf_cv,
    X_train,
    y_train,
    cv=3,
    return_train_score=True
)

print('평균 train accuracy : ', results['train_score'].mean())
print('평균 validation accuracy : ', results['test_score'].mean())

평균 train accuracy :  1.0
평균 validation accuracy :  0.9516091553386778


## 03. OOB Score

Random Forest는 부트스트랩 샘플링을 사용하기 때문에, 각 트리가 학습하지 않은 샘플이 자연스럽게 생긴다.
이 샘플을 OOB(Out-Of-Bag) 샘플이라고 한다.

OOB 점수는:
- 각 샘플을 학습하지 않은 트리들만 모아 예측하고
- 그 예측을 바탕으로 일반화 성능을 추정하는 방식이다

즉, 별도의 검증 세트를 따로 만들지 않고도 훈련 데이터 내부에서 검증 비슷한 평가를 할 수 있다.

In [11]:
rf_clf_oob = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    oob_score=True,
    random_state=42
)

rf_clf_oob.fit(X_train, y_train)

print('train score', rf_clf_oob.score(X_train, y_train))
print('OOB score', rf_clf_oob.oob_score_)
print('oob_decision_function_.shape', rf_clf_oob.oob_decision_function_.shape)

# oob 확률 확인
oob_proba_df = pd.DataFrame(
    rf_clf_oob.oob_decision_function_,
    columns=['class_0_proba', 'class_1_proba']
)
oob_proba_df

train score 0.9934065934065934
OOB score 0.9626373626373627
oob_decision_function_.shape (455, 2)


,class_0_proba,class_1_proba
0,0.001831,0.998169
1,0.999449,0.000551
2,0.007297,0.992703
3,0.010121,0.989879
4,0.001742,0.998258
...,...,...
450,0.534073,0.465927
451,1.000000,0.000000
452,0.967742,0.032258
453,1.000000,0.000000


## 04. RandomForestRegressor

회귀에서는 분류와 결합 방식이 다르다.

- 분류: 여러 트리의 클래스 판단을 결합
- 회귀: 여러 트리의 예측값을 평균

즉, 회귀에서는 "다수결"이 아니라 "평균"이 핵심이다.

In [12]:
# 캘리포니아 주택 가격 데이터 로드
df = pd.read_csv('data/california_housing.csv')

# 데이터 구조 확인
print(df.head())
print(df.info())

# 타겟과 피처 분리
X = df.drop('MedHouseVal', axis=1).to_numpy()
y = df['MedHouseVal'].to_numpy()

# 훈련/테스트 분리
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

   MedInc  HouseAge  AveRooms  AveBedrms  Population  AveOccup  Latitude  \
0  8.3252      41.0  6.984127   1.023810       322.0  2.555556     37.88   
1  8.3014      21.0  6.238137   0.971880      2401.0  2.109842     37.86   
2  7.2574      52.0  8.288136   1.073446       496.0  2.802260     37.85   
3  5.6431      52.0  5.817352   1.073059       558.0  2.547945     37.85   
4  3.8462      52.0  6.281853   1.081081       565.0  2.181467     37.85   

   Longitude  MedHouseVal  
0    -122.23        4.526  
1    -122.22        3.585  
2    -122.24        3.521  
3    -122.25        3.413  
4    -122.25        3.422  
<class 'pandas.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   MedInc       20640 non-null  float64
 1   HouseAge     20640 non-null  float64
 2   AveRooms     20640 non-null  float64
 3   AveBedrms    20640 non-null  float64
 4   Population   20640 no

In [13]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

rf_reg = RandomForestRegressor(
    n_estimators=300,
    max_depth=5,
    random_state=42
)

rf_reg.fit(X_train, y_train)

train_pred = rf_reg.predict(X_train)
test_pred = rf_reg.predict(X_test)

print('train r^2', rf_reg.score(X_train, y_train))
print('test r^2', rf_reg.score(X_test, y_test))
print('train mse:', mean_squared_error(y_train, train_pred))
print('test mse:', mean_squared_error(y_test, test_pred))

train r^2 0.6819736189774857
test r^2 0.6468435714612639
train mse: 0.4251307369251397
test mse: 0.46277935468065035


## 05. 하이퍼파라미터 튜닝

Random Forest는 기본 성능이 나쁘지 않은 편이지만, 다음과 같은 하이퍼파라미터를 조정하면 일반화 성능이 달라질 수 있다.

- n_estimators: 트리 개수
- max_depth: 트리 최대 깊이
- min_samples_split: 분할을 위한 최소 샘플 수
- min_samples_leaf: 리프 노드의 최소 샘플 수

In [15]:
from sklearn.model_selection import GridSearchCV

# 하이퍼파라미터 최적화
param_grid = {
    'n_estimators': [300, 500],
    'max_depth': [5, 7, 9],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 3]
}

grid_search = GridSearchCV(
    RandomForestRegressor(random_state=42),
    param_grid=param_grid,
    cv=3,
    scoring='r2',
    n_jobs=-1
)
grid_search.fit(X_train, y_train)

print('Best params:', grid_search.best_params_)
print('Best CV score:', grid_search.best_score_)

KeyboardInterrupt: 

In [ ]:
# 최적 모델로 테스트 성능 확인
best_rf_reg = grid_search.best_estimator_
best_test_pred = best_rf_reg.predict(X_test)

print('Best model Test R2:', best_rf_reg.score(X_test, y_test))
print('Best model Test MSE:', mean_squared_error(y_test, best_test_pred))

## 06. 정리

1. Bagging은 같은 유형의 모델을 여러 개 학습시켜 예측을 결합하는 방식이다.
2. Random Forest는 Bagging에 특성 무작위 선택을 더한 대표 모델이다.
3. 분류에서는 클래스 판단을 결합하고, 회귀에서는 예측값을 평균한다.
4. OOB 점수는 부트스트랩 샘플링 덕분에 가능한 내부 검증 방식이다.
5. Random Forest는 앙상블을 처음 배울 때 가장 좋은 출발점 중 하나이다.